# $H_a$ family: nonlinearity, Walsh spectrum, APN/permutation verification, and affine equivalence

This notebook parallels the $G_a$ notebook but studies the **second generalised Li-Kaleyski family**

$$H_a(x,y,z) = \bigl(x^{q+1}+axy^q+yz^q,\; xy^q+z^{q+1},\; x^qz+y^{q+1}+ay^qz\bigr),\quad a\in\mathbb{F}_{2^m}^*,\quad q=2^i,\quad \gcd(i,m)=1,\quad m \text{ odd.}$$

**Governing polynomial** (Theorem 3.1 of the paper):
$$R_a(T) = T^{q^2+q+1}+(aT+1)^{q+1}.$$
$H_a$ is a permutation $\iff$ $H_a$ is APN $\iff$ $R_a$ has no root in $\mathbb{F}_{2^m}$.

Since $R_a$ is root-equivalent to $Q_a = T^{q^2+q+1}+aT+1$ (Proposition 1.1), the **same values of $a$** make both $G_a$ and $H_a$ APN permutations.

**Diagonal equivalence** (analogue of Proposition 2.6, derived by matching the 8 monomial equations F1-F8 for $H_1 = A_1\circ H_a\circ A_2$): the condition is $a^{q^2+q+1}=1$ — identical to the $G_a$ case.

**Cells:**
1. Exact nonlinearity of $H_1$
2. Walsh-spectrum comparison across all good $a$ (necessary condition for affine equivalence)
3. Permutation / APN / root-condition verification table
4. Diagonal equivalence check ($a^{q^2+q+1}=1$, with explicit witnesses)
5. Full $\mathbb{F}_2$-linear affine equivalence check between $H_a$ and $H_1$

In [ ]:
from sage.all import *
from sage.crypto.boolean_function import BooleanFunction
import time

def compute_nl_H1(m, i):
    F  = GF(2**m, 'w')
    q  = 2**i
    print(f'--- Exact Nonlinearity of H_1  (m={m}, i={i}, q={q}) ---')
    bits = []
    for x in F:
        xq = x**q; xq1 = x*xq
        for y in F:
            yq = y**q; yq1 = y*yq
            for z in F:
                zq = z**q; zq1 = z*zq
                # H_1: a=1
                c1 = xq1 + x*yq + y*zq
                c2 = x*yq + zq1
                c3 = xq*z + yq1 + yq*z
                bits.append(int((c1 + c2 + c3).trace()))
    bf = BooleanFunction(bits)
    nl = bf.nonlinearity()
    print(f'Exact Nonlinearity of H_1: {nl}')
    return nl

compute_nl_H1(m=7, i=1)


In [ ]:
from sage.all import *
from sage.crypto.boolean_function import BooleanFunction
import time

def get_invariants_Ha(m, i, a_val):
    F = GF(2**m, 'w'); q = 2**i
    bits = []
    for x in F:
        xq=x**q; xq1=x*xq
        for y in F:
            yq=y**q; yq1=y*yq
            for z in F:
                zq=z**q; zq1=z*zq
                c1 = xq1 + a_val*x*yq + y*zq
                c2 = x*yq + zq1
                c3 = xq*z + yq1 + a_val*yq*z
                bits.append(int((c1+c2+c3).trace()))
    bf = BooleanFunction(bits)
    return bf.nonlinearity(), tuple(sorted(bf.walsh_hadamard_transform()))

def run_walsh_comparison_Ha(m, i):
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1
    P.<T> = F[]
    print(f'--- Walsh-spectrum comparison for H_a  (m={m}, i={i}, q={q}) ---')
    print('Computing invariants for H_1 ...')
    t0 = time.time()
    nl1, spec1 = get_invariants_Ha(m, i, F(1))
    print(f'H_1: NL={nl1}  ({time.time()-t0:.1f}s)\n')
    print(f'{"a":<20} {"NL":<10} {"Spec=H_1?":<12} {"time":>6}')
    print('-'*52)
    for a in F:
        if a == 0 or a == 1: continue
        Qa = T**d + a*T + 1
        if Qa.roots(): continue   # skip bad a
        t0 = time.time()
        nla, speca = get_invariants_Ha(m, i, a)
        match = (nl1==nla and spec1==speca)
        tag = 'MATCH' if match else 'DIFFERENT'
        print(f'{str(a):<20} {nla:<10} {tag:<12} {time.time()-t0:>5.1f}s')

run_walsh_comparison_Ha(m=7, i=1)


In [ ]:
from sage.all import *
import time

def build_Ha_lut(m, i, a_val):
    F = GF(2**m, 'w'); q = 2**i
    lut = {}
    for x in F:
        xq=x**q; xq1=x*xq
        for y in F:
            yq=y**q; yq1=y*yq
            for z in F:
                zq=z**q; zq1=z*zq
                c1 = xq1 + a_val*x*yq + y*zq
                c2 = x*yq + zq1
                c3 = xq*z + yq1 + a_val*yq*z
                lut[(x,y,z)] = (c1,c2,c3)
    return lut

def check_APN_Ha(m, i, a_val):
    F = GF(2**m, 'w'); lut = build_Ha_lut(m, i, a_val)
    triples = list(lut.keys())
    for (A,B,C) in triples:
        if A==B==C==F(0): continue
        cnt = sum(1 for (x,y,z) in triples
                  if (lut[(x+A,y+B,z+C)][0]+lut[(x,y,z)][0]==lut[(A,B,C)][0] and
                      lut[(x+A,y+B,z+C)][1]+lut[(x,y,z)][1]==lut[(A,B,C)][1] and
                      lut[(x+A,y+B,z+C)][2]+lut[(x,y,z)][2]==lut[(A,B,C)][2]))
        if cnt > 2: return False
    return True

def verify_table_Ha(m, i):
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1
    P.<T> = F[]
    d0 = gcd(d, 2**m-1)
    print(f'm={m}, i={i}, q={q},  q^2+q+1={d},  2^m-1={2**m-1},  d0={d0}')
    print(f'Predicted diag-linear-equiv: a^(q^2+q+1)=1  ({d0} element(s))')
    print(f'Checking: FULL AFFINE  [N=2^{{3*m}}={2**(3*m)}]')
    print('='*72)
    print(f'H_1 reference built.')
    print()
    hdr = f'{"a":<20} {"Ra root-free?":<16} {"perm?":<8} {"APN?":<8} {"time":>6}'
    print(hdr); print('-'*62)
    for a in F:
        if a == 0: continue
        t0 = time.time()
        Qa = T**d + a*T + 1
        no_root = (len(Qa.roots()) == 0)
        lut = build_Ha_lut(m, i, a)
        is_perm = (len(set(lut.values())) == 2**(3*m))
        is_apn = check_APN_Ha(m, i, a) if m <= 5 else 'skipped'
        consistent = (no_root == is_perm == (is_apn if isinstance(is_apn, bool) else no_root))
        print(f'{str(a):<20} {str(no_root):<16} {str(is_perm):<8} '
              f'{str(is_apn):<8} {time.time()-t0:>5.1f}s   {"OK" if consistent else "MISMATCH!"}')

verify_table_Ha(m=3, i=1)
print()
verify_table_Ha(m=5, i=1)


In [ ]:
from sage.all import *

def check_diagonal_equiv_Ha(m, i):
    # H_1 = A1 o H_a o A2 for diagonal maps A1(u,v,w)=(mu*u,nu*v,rho*w),
    # A2(x,y,z)=(lam1*x,lam2*y,lam3*z).  Matching the 8 monomials gives:
    # F1: mu*lam1^{q+1}=1   F2: mu*a*lam1*lam2^q=1   F3: mu*lam2*lam3^q=1
    # F4: nu*lam1*lam2^q=1  F5: nu*lam3^{q+1}=1
    # F6: rho*lam1^q*lam3=1 F7: rho*lam2^{q+1}=1     F8: rho*a*lam2^q*lam3=1
    # Eliminating mu,nu,rho and deriving consistency yields a^{q^2+q+1}=1.
    F = GF(2**m, 'w'); q = 2**i; d = q**2+q+1; d0 = gcd(d, 2**m-1)
    print(f'=== Diagonal equivalence H_a ~ H_1  (m={m}, i={i}, q={q}) ===')
    print(f'Predicted condition: a^{d}=1,  subgroup order d0={d0}')
    nonzero = [e for e in F if e != 0]

    def H1():
        lut = {}
        for x in F:
            xq=x**q; xq1=x*xq
            for y in F:
                yq=y**q; yq1=y*yq
                for z in F:
                    zq=z**q; zq1=z*zq
                    lut[(x,y,z)]=(xq1+x*yq+y*zq, x*yq+zq1, xq*z+yq1+yq*z)
        return lut

    lut1 = H1()
    diag_equiv = []
    for a in F:
        if a == 0: continue
        pred = (a**d == F(1))
        found = False
        for lam2 in nonzero:
            lam1 = a**(-(q+1)) * lam2
            lam3 = lam2 / a
            mu = lam1**(-(q+1)); nu = lam3**(-(q+1)); rho = lam2**(-(q+1))
            if not (mu*lam1**(q+1)==1 and mu*a*lam1*lam2**q==1 and
                    mu*lam2*lam3**q==1 and nu*lam1*lam2**q==1 and
                    nu*lam3**(q+1)==1 and rho*lam1**q*lam3==1 and
                    rho*lam2**(q+1)==1 and rho*a*lam2**q*lam3==1):
                continue
            Ha_lut = {}
            for x in F:
                xq=x**q; xq1=x*xq
                for y in F:
                    yq=y**q; yq1=y*yq
                    for z in F:
                        zq=z**q; zq1=z*zq
                        Ha_lut[(x,y,z)]=(xq1+a*x*yq+y*zq, x*yq+zq1, xq*z+yq1+a*yq*z)
            err = sum(1 for (x,y,z) in Ha_lut
                      if (mu*Ha_lut[(lam1*x,lam2*y,lam3*z)][0],
                          nu*Ha_lut[(lam1*x,lam2*y,lam3*z)][1],
                          rho*Ha_lut[(lam1*x,lam2*y,lam3*z)][2]) != lut1[(x,y,z)])
            if err == 0: found = True; break
        if found: diag_equiv.append(a)
        match = 'OK' if (pred==found) else 'MISMATCH!'
        print(f'  a={str(a):<20}  a^d=1: {str(pred):<6}  witness: {str(found):<6}  {match}')
    print(f'Diagonal-equiv elements ({len(diag_equiv)}): {[str(a) for a in diag_equiv]}')
    print()

check_diagonal_equiv_Ha(m=3, i=1)
check_diagonal_equiv_Ha(m=5, i=1)


In [ ]:
from sage.all import *
import time

# Gaussian elimination helpers for partial F_2-linear map
def la(rows, d, i):
    rows = list(rows)
    for rd,ri in rows:
        if (d>>(rd.bit_length()-1))&1: d^=rd; i^=ri
    if d==0: return (i==0), rows
    rows.append((d,i)); return True, rows

def lq(rows, d):
    i=0
    for rd,ri in rows:
        if (d>>(rd.bit_length()-1))&1: d^=rd; i^=ri
    return i if d==0 else None

def lab(rows, items):
    for d,i in items.items():
        ok,rows = la(rows,d,i)
        if not ok: return False,None
    return True, rows

def leval(rows, NN):
    lut=[0]*NN
    for x in range(1,NN):
        hb=x.bit_length()-1; le=lq(rows,1<<hb)
        if le is None: return None
        lut[x]=lut[x^(1<<hb)]^le
    return lut

def check_linear_equiv(F_lut, G_lut, n):
    NN=1<<n; basis=[1<<k for k in range(n)]
    def is_lin(lt):
        mat=[lt[basis[k]] for k in range(n)]
        for x in range(NN):
            ex=0
            for k in range(n):
                if x&basis[k]: ex^=mat[k]
            if lt[x]!=ex: return False
        return True
    for v0 in range(1,NN):
        fv0=F_lut[v0]
        if fv0==0: continue
        ok,rows=la([],0,G_lut[0])
        if not ok: continue
        ok,rows=la(rows,fv0,G_lut[basis[0]])
        if not ok: continue
        sp=[(0,0),(basis[0],v0)]; good=True
        for j in range(1,n):
            outs={xo for _,xo in sp}; placed=False
            for w in range(1,NN):
                if w in outs: continue
                ne={}; cf=False
                for si,so in sp:
                    fx=F_lut[w^so]; gx=G_lut[basis[j]^si]
                    qv=lq(rows,fx)
                    if qv is not None:
                        if qv!=gx: cf=True; break
                    elif fx in ne:
                        if ne[fx]!=gx: cf=True; break
                    else: ne[fx]=gx
                if cf: continue
                ok2,r2=lab(rows,ne)
                if ok2:
                    rows=r2; sp=sp+[(basis[j]^si,w^so) for si,so in sp]
                    placed=True; break
            if not placed: good=False; break
        if not good: continue
        L2=[0]*NN
        for xi,xo in sp: L2[xi]=xo
        if len(set(L2))!=NN: continue
        L1=leval(rows,NN)
        if L1 is None: continue
        if not all(L1[F_lut[L2[x]]]==G_lut[x] for x in range(NN)): continue
        if not is_lin(L2): continue
        return True, L2, L1
    return False, None, None

def build_int_luts_Ha(m, i, a_sage, F):
    q=2**i; N=2**m; elts=list(F)
    idx={e:k for k,e in enumerate(elts)}
    def ti(x,y,z): return idx[x]|(idx[y]<<m)|(idx[z]<<(2*m))
    Ha={}; H1={}
    a=a_sage
    for x in elts:
        xq=x**q; xq1=x*xq
        for y in elts:
            yq=y**q; yq1=y*yq
            for z in elts:
                zq=z**q; zq1=z*zq; key=ti(x,y,z)
                Ha[key]=ti(xq1+a*x*yq+y*zq, x*yq+zq1, xq*z+yq1+a*yq*z)
                H1[key]=ti(xq1+x*yq+y*zq,   x*yq+zq1, xq*z+yq1+yq*z)
    return Ha, H1

def shifted(lut, c2, NN):
    gc2=lut[c2]
    return {x: lut[x^c2]^gc2 for x in range(NN)}

def check_affine_equiv_Ha(m, i, full_affine=True):
    F=GF(2**m,'w'); q=2**i; d=q**2+q+1; d0=gcd(d,2**m-1)
    NN=2**(3*m); n=3*m; P.<T>=F[]
    print(f'm={m}, i={i}, q={q},  q^2+q+1={d},  2^m-1={2**m-1},  d0={d0}')
    mode='FULL AFFINE' if full_affine else 'LINEAR ONLY'
    print(f'Predicted diag-linear-equiv: a^(q^2+q+1)=1  ({d0} element(s))')
    print(f'Checking: {mode}  [N=2^{n}={2**n}]')
    print('='*72)
    t_build=time.time()
    _,H1d=build_int_luts_Ha(m,i,F(1),F)
    H1=[H1d[x] for x in range(NN)]
    print(f'H_1 lookup built in {time.time()-t_build:.2f}s\n')
    hdr=f'{"a":<20} {"a^d=1?":<10} {"perm?":<8} {"APN?":<6} {"affine~H_1?":<13} {"consistent?":<13} {"time":>6}'
    print(hdr); print('-'*80)
    for a_sage in F:
        if a_sage==0: continue
        t0=time.time()
        Qa=T**d+a_sage*T+1; no_root=(len(Qa.roots())==0)
        diag_eq=(a_sage**d==F(1))
        Had,_=build_int_luts_Ha(m,i,a_sage,F)
        Ha=[Had[x] for x in range(NN)]
        is_perm=(len(set(Ha))==NN)
        if full_affine:
            found=False
            for c2 in range(NN):
                Fc2=shifted(Ha,c2,NN)
                ok,_,_=check_linear_equiv(Fc2,H1,n)
                if ok: found=True; break
        else:
            ok,_,_=check_linear_equiv(Ha,H1,n); found=ok
        consistent=(no_root==is_perm==found)
        print(f'{str(a_sage):<20} {str(diag_eq):<10} {str(is_perm):<8} '
              f'{str(no_root):<6} {str(found):<13} {str(consistent):<13} {time.time()-t0:>5.1f}s')
    print('-'*80)

check_affine_equiv_Ha(m=3, i=1, full_affine=True)
print()
check_affine_equiv_Ha(m=5, i=1, full_affine=True)
